# PaySim: Phương Pháp Tối Ưu Toàn Diện
1. **Tri-Model Ensemble**: Kết hợp XGBoost + LightGBM + CatBoost
2. **Cost-Sensitive Threshold Tuning**: Tối ưu ngưỡng quyết định theo thiệt hại kinh tế
3. **Giải thích mô hình bằng SHAP**


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix, precision_recall_curve
)
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier
from ipynb.fs.defs.preprocessing import get_data
from util import summary


ImportError: Could not import e:\HUST\CLB\GDG\Nhom01_GDG-main\notebooks\paysim\preprocessing.ipynb for ipynb.fs.defs.preprocessing: not a valid ipynb file

In [ ]:
X_train, X_test, y_train, y_test = get_data(sample_size=150000, test_size=0.3)
pos_weight = (len(y_train) - y_train.sum()) / y_train.sum()

print("Huấn luyện 3 mô hình Gradient Boosting...")
xgb_clf = xgb.XGBClassifier(n_estimators=150, max_depth=6, learning_rate=0.05, scale_pos_weight=pos_weight, eval_metric="aucpr", random_state=42, n_jobs=-1)
xgb_clf.fit(X_train, y_train)

lgb_clf = lgb.LGBMClassifier(n_estimators=150, num_leaves=63, learning_rate=0.05, scale_pos_weight=pos_weight, random_state=42, n_jobs=-1, verbose=-1)
lgb_clf.fit(X_train, y_train)

cb_clf = CatBoostClassifier(iterations=150, depth=6, learning_rate=0.05, auto_class_weights="Balanced", random_seed=42, verbose=0)
cb_clf.fit(X_train, y_train)


In [ ]:
p_xgb = xgb_clf.predict_proba(X_test)[:, 1]
p_lgb = lgb_clf.predict_proba(X_test)[:, 1]
p_cb  = cb_clf.predict_proba(X_test)[:, 1]

p_ensemble = 0.4 * p_xgb + 0.4 * p_lgb + 0.2 * p_cb

print("--- Kết quả Ensemble ở ngưỡng mặc định 0.5 ---")
summary(y_test, (p_ensemble >= 0.5).astype(int))


In [ ]:
# Quét ngưỡng tối ưu chi phí (FN=$200, FP=$10)
C_FN, C_FP = 200, 10
thresholds = np.linspace(0.05, 0.95, 100)
costs = []
for th in thresholds:
    pred = (p_ensemble >= th).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()
    costs.append(fn * C_FN + fp * C_FP)

best_idx = int(np.argmin(costs))
opt_th = float(thresholds[best_idx])
print("Ngưỡng tối ưu T* =", round(opt_th, 3))
print("Tổn thất tối thiểu ước tính: $" + str(int(costs[best_idx])))

summary(y_test, (p_ensemble >= opt_th).astype(int))


In [ ]:
import shap
explainer = shap.TreeExplainer(lgb_clf)
sample_X = X_test.iloc[:200]
shap_values = explainer.shap_values(sample_X)
vals = shap_values[1] if isinstance(shap_values, list) else shap_values
shap.summary_plot(vals, sample_X, max_display=10)
